# 01 ? Data pipeline

Audit and clean the configured BTC-USDT dataset, then save reusable Parquet data and a small set of exploratory figures. Run notebooks **01 ? 04** in order. The 800,000-row cap is intentional; raise it only when memory and Spark capacity permit.

In [ ]:
## Setup
import os

import sys, warnings, yaml
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
_candidates.extend(path.parent for path in _cwd.glob("*/configs/pipeline.yaml"))
PROJECT_ROOT = next((path for path in _candidates if (path / "configs" / "pipeline.yaml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find configs/pipeline.yaml. Open the project folder or set the notebook working directory to this repository.")
os.chdir(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline.yaml"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.spark_session import get_spark
from src.data.loader        import primary_to_pandas
from src.data.profiler      import profile_dataframe
from src.data.cleaner       import standardise_timestamps, clean_and_align, report_gaps
from src.visualisation.eda_plots import plot_price_volume, plot_returns_distribution

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
for d in [PROCESSED, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print('Setup complete.')

In [ ]:
# ── 1b. Materialise full primary dataset ─────────────────────────────
print('Loading primary dataset (up to 800,000 rows)...')
df_primary = primary_to_pandas(max_rows=800_000)
print(f'Sample shape: {df_primary.shape}')
sample = df_primary.head(10_000)
null_pct = (sample.isnull().sum() / len(sample) * 100).round(2)
print(f'\nNull % (sample):')
flagged = null_pct[null_pct > 0]
print(flagged.to_string() if len(flagged) else '  None.')
print(f'Duplicates (sample): {sample.duplicated().sum():,}')

In [ ]:
# ── 1c. Timestamp detection & coverage ───────────────────────────────
TS_CANDIDATES = [c for c in df_primary.columns
                 if c.lower() in ('timestamp','date','open_time','time','close_time')]
TS_COL = TS_CANDIDATES[0] if TS_CANDIDATES else df_primary.columns[0]
print(f'Timestamp column : "{TS_COL}"')

try:
    df_primary[TS_COL] = pd.to_datetime(df_primary[TS_COL], unit='ms', utc=True)
except Exception:
    df_primary[TS_COL] = pd.to_datetime(df_primary[TS_COL], utc=True)

df_primary = df_primary.sort_values(TS_COL)
median_delta    = df_primary[TS_COL].diff().median()
EXPECTED_SECS   = int(median_delta.total_seconds()) if hasattr(median_delta,'total_seconds') else 3600

print(f'Date range  : {df_primary[TS_COL].min()}  →  {df_primary[TS_COL].max()}')
print(f'Total bars  : {len(df_primary):,}')
print(f'Bar interval: {median_delta}  ({EXPECTED_SECS}s)')

In [ ]:
# ── 1d. Tick / order-book column check ───────────────────────────────
TICK_INDICATORS = [
    'taker_buy_base_asset_volume','taker_buy_quote_asset_volume',
    'number_of_trades','trade_id','aggressor_side','bid_volume','ask_volume',
]
found_tick_cols = [c for c in df_primary.columns if c in TICK_INDICATORS]
HAS_TICK_DATA   = len(found_tick_cols) > 0

print(f'Tick / order-book columns : {found_tick_cols}')
print(f'Hawkes / OFI / VPIN active: {HAS_TICK_DATA}')
if not HAS_TICK_DATA:
    print('  → Conditional microstructure features are OUT OF SCOPE for this dataset.')

In [ ]:
# ── 1f. Save audit metadata ───────────────────────────────────────────
OHLC_COLS  = [c for c in ['open','high','low','close'] if c in df_primary.columns]
audit_meta = {
    'ts_col'                    : TS_COL,
    'has_tick_data'             : bool(HAS_TICK_DATA),
    'tick_columns'              : found_tick_cols,
    'expected_interval_seconds' : EXPECTED_SECS,
    'date_min'                  : str(df_primary[TS_COL].min()),
    'date_max'                  : str(df_primary[TS_COL].max()),
    'total_bars'                : int(len(df_primary)),
    'ohlc_columns'              : OHLC_COLS,
    'volume_col'                : 'volume' if 'volume' in df_primary.columns else None,
    'null_pct'                  : null_pct.to_dict(),
    'use_secondary'             : USE_SECONDARY,
}
meta_path = PROCESSED / 'audit_meta.yaml'
with open(meta_path, 'w') as f:
    yaml.dump(audit_meta, f, default_flow_style=False)
print(f'Audit metadata saved → {meta_path}')

---
## 2 · PySpark Ingestion & Cleaning

In [ ]:
# ── 2a. Start Spark ───────────────────────────────────────────────────
spark = get_spark()
print(f'Spark {spark.version} started.')

In [ ]:
# ── 2b. Load full dataset → Spark DataFrame ──────────────────────────
# Load all rows via Arrow (non-streaming) for the full training corpus.
print('Loading full dataset via Arrow (non-streaming)…')
df_full = primary_to_pandas(max_rows=800_000)  # Adjust max_rows as needed for your dataset
print(f'Full dataset shape: {df_full.shape}')

# The timestamp column is a datetime string ('YYYY-MM-DD HH:MM:SS').
# standardise_timestamps() auto-detects the string type and uses
# F.to_timestamp() — no epoch-ms conversion needed here.
sdf = spark.createDataFrame(df_full)
print(f'Spark DataFrame : {sdf.count():,} rows × {len(sdf.columns)} columns')
sdf.printSchema()

In [ ]:
# ── 2c. Profile raw data ──────────────────────────────────────────────
_ = profile_dataframe(sdf, name='Primary Raw')

In [ ]:
# ── 2d. Standardise + Clean ───────────────────────────────────────────
sdf_clean = standardise_timestamps(sdf, ts_col=TS_COL)
for column in ('open', 'high', 'low', 'close', 'volume'):
    if column in sdf_clean.columns:
        sdf_clean = sdf_clean.withColumn(column, F.col(column).cast('double'))
sdf_clean = sdf_clean.dropna(subset=[TS_COL, 'open', 'high', 'low', 'close'])
sdf_clean = sdf_clean.filter(F.col('close') > 0)
sdf_clean = clean_and_align(sdf_clean, ts_col=TS_COL)

n_before = sdf.count()
n_after  = sdf_clean.count()
print(f'After cleaning: {n_after:,} rows  (dropped {n_before - n_after:,})')


In [ ]:
# ── 2e. Timestamp gap report ─────────────────────────────────────────
gaps_sdf  = report_gaps(sdf_clean, ts_col=TS_COL, expected_interval_seconds=EXPECTED_SECS)
gap_count = gaps_sdf.count()
print(f'Gaps larger than {EXPECTED_SECS}s : {gap_count:,}')
if gap_count > 0:
    gaps_sdf.orderBy('gap_seconds', ascending=False).show(20, truncate=False)

In [ ]:
# ── 2f. Collect to pandas + save ─────────────────────────────────────
df_clean = sdf_clean.toPandas()
df_clean[TS_COL] = pd.to_datetime(df_clean[TS_COL])
df_clean = df_clean.sort_values(TS_COL).reset_index(drop=True)

out_path = PROCESSED / 'btc_cleaned.parquet'
df_clean.to_parquet(out_path, index=False)
print(f'Saved → {out_path}  ({df_clean.shape})')

spark.stop()
print('Spark stopped.')

---
## 3 · Exploratory Data Analysis (EDA)

In [ ]:
# ── 3a. Price & Volume ────────────────────────────────────────────────
fig = plot_price_volume(df_clean, ts_col=TS_COL)
fig.write_html(str(FIGURES / 'price_volume.html'), include_plotlyjs='cdn')


In [ ]:
# ── 3b. Log-return descriptives ───────────────────────────────────────
df_clean['log_return'] = np.log(df_clean['close'] / df_clean['close'].shift(1)).replace([np.inf, -np.inf], np.nan)
returns = df_clean['log_return'].dropna()

print(returns.describe().round(6))
print(f'Skewness : {returns.skew():.4f}  |  Kurtosis (excess): {returns.kurtosis():.4f}')

fig = plot_returns_distribution(df_clean)
fig.write_html(str(FIGURES / 'return_distribution.html'), include_plotlyjs='cdn')


In [ ]:
# ── 3c. Volatility clustering + rolling vol ───────────────────────────
fig = go.Figure(go.Scatter(
    x=df_clean[TS_COL], y=df_clean['log_return']**2,
    line=dict(color='#F7931A', width=0.7), name='Squared Return',
))
fig.update_layout(title='Volatility Clustering (Squared Log Returns)',
                  template='plotly_dark', yaxis_title='Squared Return')
fig.write_html(str(FIGURES / 'vol_clustering.html'), include_plotlyjs='cdn')

fig2 = go.Figure()
for w, col in [(7,'#4FC3F7'), (30,'#F7931A'), (90,'#66BB6A')]:
    fig2.add_trace(go.Scatter(x=df_clean[TS_COL], y=df_clean['log_return'].rolling(w).std(),
                              name=f'{w}-bar vol', line=dict(color=col)))
fig2.update_layout(title='Rolling Volatility', template='plotly_dark')
fig2.write_html(str(FIGURES / 'rolling_vol_eda.html'), include_plotlyjs='cdn')
fig2.show()

In [ ]:
# ── 3d. ACF / PACF ───────────────────────────────────────────────────
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
plot_acf(returns,     lags=50, ax=axes[0,0], title='ACF – Log Returns')
plot_pacf(returns,    lags=50, ax=axes[0,1], title='PACF – Log Returns')
plot_acf(returns**2,  lags=50, ax=axes[1,0], title='ACF – Squared Returns')
plot_pacf(returns**2, lags=50, ax=axes[1,1], title='PACF – Squared Returns')
plt.tight_layout()
plt.savefig(str(FIGURES / 'acf_pacf.png'), dpi=120, bbox_inches='tight')
plt.close()

print('EDA complete.')